In [ ]:
# ▶ 0. 필수 라이브러리 설치 및 임포트
!pip install openpyxl xlrd
import pandas as pd
import os
import re
import zipfile
from glob import glob

# ▶ 1. 매핑용 ZIP 압축 해제
def unzip_file(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"✅ 압축 해제 완료: {extract_to}")

unzip_file("/content/14~23 주,야간 정리.zip", "/content/14_23")

# ▶ 2. 학과명 정제 함수 정의
def clean_department_name(name):
    if pd.isna(name): return name
    name = str(name)
    name = re.sub(r"\(\d+\)", "", name)             # 학과코드 제거
    name = re.sub(r"[・ㆍ·‧․.•∙･]", "", name)        # 가운뎃점 제거
    name = name.replace(" ", "")                    # 공백 제거
    return name.strip()

# ▶ 3. 연도별 계열 매핑 딕셔너리 생성
year_to_mapping = {}

for path in glob("/content/14_23/*.xls*"):
    try:
        df_map = pd.read_excel(path)
        if '학과명' in df_map.columns and '대계열' in df_map.columns and '조사년도' in df_map.columns:
            year = int(df_map['조사년도'].dropna().iloc[0])
            temp = df_map[['학과명', '대계열']].dropna().copy()
            temp['학과명'] = temp['학과명'].apply(clean_department_name)
            temp['대계열'] = temp['대계열'].astype(str).str.strip()
            mapping = dict(zip(temp['학과명'], temp['대계열']))
            if year in year_to_mapping:
                year_to_mapping[year].update(mapping)
            else:
                year_to_mapping[year] = mapping
    except Exception as e:
        print(f"❌ {os.path.basename(path)} → {e}")

# ▶ 4. 전처리할 대상 파일 설정 (졸업생 진학 현황)
target_path = "/content/2018년 _대학_5-다. 졸업생의 취업 현황_학과별자료.xlsx"
df = pd.read_excel(target_path, header=3)  # 실제 데이터 시작 행부터

# ▶ 5. NaN 컬럼 헤더 정리 및 학과명 자동 탐색
columns = [str(c).strip() if 'Unnamed' not in str(c) else None for c in df.columns]
for i in range(len(columns)):
    if columns[i] is None and i > 0:
        columns[i] = columns[i - 1]
df.columns = columns

# ▶ 6. 병합된 셀 해제 및 채우기
for col in ['기준년도', '학교명', '단과대학', '학과(전공)']:
    if col in df.columns:
        df[col] = df[col].ffill()

# ▶ 7. 학과명 정제 및 필터링
dept_col = next((col for col in df.columns if "학과" in col), None)
if not dept_col:
    raise ValueError(f"❌ '학과'가 포함된 열을 찾을 수 없습니다.\n🧪 현재 컬럼들: {df.columns.tolist()}")

df[dept_col] = df[dept_col].apply(clean_department_name)

# ▶ 8. 사이버대학 / 야간·원격 제거
if '학교명' in df.columns:
    df = df[~df['학교명'].str.contains("사이버대학", na=False)]
if '구분' in df.columns:
    df = df[~df['구분'].isin(['원격'])]

# ▶ 9. 연도 추출 및 계열 매핑
df['기준년도'] = pd.to_numeric(df['기준년도'], errors='coerce')
def match_by_year(row):
    year = int(row['기준년도']) if not pd.isna(row['기준년도']) else None
    dept = row[dept_col]
    mapping = year_to_mapping.get(year, {})
    return mapping.get(dept, None)

df['계열'] = df.apply(match_by_year, axis=1)
df['계열'] = df['계열'].fillna("미분류")

# ▶ 10. 결과 저장
output_path = "/content/2018년_진학현황_계열분류완료.xlsx"
df.to_excel(output_path, index=False)
print(f"✅ 최종 저장 완료 → {output_path}")


✅ 압축 해제 완료: /content/14_23


In [ ]:
import pandas as pd

# ▶ 1. 파일 로딩
file_path = "/content/2018년_진학현황_계열분류완료.xlsx"
df = pd.read_excel(file_path)

# ▶ 2. 열 이름 자동 탐색
def find_col(keyword):
    return [col for col in df.columns if keyword in str(col)]

grad_cols = find_col("졸업자(A)")
adv_cols = find_col("진학자(C)")
mil_cols = find_col("입대자(D)")
emp_cols = find_col("취업자(B)")
impossible_cols = find_col("취업불가능자(E)")
foreigner_cols = find_col("외국인유학생(F)")
exclude_cols = find_col("건강보험직장가입제외대상(G)")

# ▶ 3. 필수값 점검
required = [grad_cols, adv_cols, mil_cols, emp_cols]
if not all(required):
    raise ValueError("❌ 졸업자, 진학자, 입대자, 취업자 관련 컬럼을 모두 찾을 수 없습니다.")

# ▶ 4. 남녀 합계 구하기
def sum_columns(col_list):
    return df[col_list].apply(pd.to_numeric, errors='coerce').sum(axis=1)

df['졸업자'] = sum_columns(grad_cols)
df['진학자'] = sum_columns(adv_cols)
df['입대자'] = sum_columns(mil_cols)
df['취업자'] = sum_columns(emp_cols)
df['취업불가능자'] = sum_columns(impossible_cols) if impossible_cols else 0
df['외국인유학생'] = sum_columns(foreigner_cols) if foreigner_cols else 0
df['건보제외'] = sum_columns(exclude_cols) if exclude_cols else 0

# ▶ 5. 취업률 계산
분모 = df['졸업자'] - (df['진학자'] + df['입대자'] + df['취업불가능자'] + df['외국인유학생'] + df['건보제외'])
df['취업률(%)'] = (df['취업자'] / 분모) * 100
df['취업률(%)'] = df['취업률(%)'].round(2)

# ▶ 6. 계열별 통합
group_keys = ['기준년도', '학교명', '계열']
numeric_cols = ['졸업자', '진학자', '입대자', '취업자', '취업불가능자', '외국인유학생', '건보제외']
keep_first_cols = ['학교종류', '설립구분', '지역', '상태']

agg_dict = {col: 'sum' for col in numeric_cols}
agg_dict.update({col: 'first' for col in keep_first_cols if col in df.columns})

df_grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)

# ▶ 7. 취업률 재계산
분모 = df_grouped['졸업자'] - (
    df_grouped['진학자'] + df_grouped['입대자'] +
    df_grouped['취업불가능자'] + df_grouped['외국인유학생'] + df_grouped['건보제외']
)
df_grouped['취업률(%)'] = (df_grouped['취업자'] / 분모) * 100
df_grouped['취업률(%)'] = df_grouped['취업률(%)'].round(2)

# ▶ 8. 컬럼 순서 복원
front_cols = group_keys + [col for col in keep_first_cols if col in df_grouped.columns]
value_cols = numeric_cols + ['취업률(%)']
final_cols = front_cols + value_cols
df_grouped = df_grouped[final_cols]

# ▶ 9. 저장
output_path = "/content/2018년_계열통합_취업률계산완.xlsx"
df_grouped.to_excel(output_path, index=False)
print(f"✅ 컬럼 순서까지 정렬된 결과 저장 완료 → {output_path}")


✅ 컬럼 순서까지 정렬된 결과 저장 완료 → /content/2017년_계열통합_취업률계산완.xlsx


In [11]:
import pandas as pd

# ▶ 1. 파일 불러오기
file_path = "/content/2023년_계열통합_취업률계산완.xlsx"  # 경로는 알맞게 수정하세요
df = pd.read_excel(file_path)

# ▶ 2. 진로진출률(%) = 취업률(%) + (진학자 / 졸업자) * 100
df["진로진출률(%)"] = round(df["취업률(%)"] + (df["진학자"] / df["졸업자"]) * 100, 2)

# ▶ 3. 졸업자대비진로성취(%) = (진학자 + 취업자) / 졸업자 * 100
df["졸업자대비진로성취(%)"] = round(((df["진학자"] + df["취업자"]) / df["졸업자"]) * 100, 2)

# ▶ 4. 결과 저장
output_path = "/content/2023년_계열통합_진로지표추가.xlsx"
df.to_excel(output_path, index=False)
print(f"✅ 저장 완료: {output_path}")

✅ 저장 완료: /content/2023년_계열통합_진로지표추가.xlsx


In [12]:
import pandas as pd
import os

# ▶ 1. 연도 범위 설정
years = range(2014, 2024)

# ▶ 2. 파일 경로 패턴 지정 (예: "data/2014년_계열통합_진로지표추가.xlsx")
folder_path = "./"  # 파일이 있는 폴더로 변경 가능
file_pattern = "{}년_계열통합_진로지표추가.xlsx"

# ▶ 3. 데이터프레임 통합
all_data = []

for year in years:
    file_path = os.path.join(folder_path, file_pattern.format(year))
    try:
        df = pd.read_excel(file_path)
        df["기준년도"] = year  # 혹시 빠졌거나 잘못되었을 경우 대비해서 덮어쓰기
        all_data.append(df)
        print(f"✅ 로딩 완료: {file_path}")
    except FileNotFoundError:
        print(f"❌ 파일 없음: {file_path}")
    except Exception as e:
        print(f"⚠️ 오류 발생 ({file_path}): {e}")

# ▶ 4. 하나의 데이터프레임으로 병합
merged_df = pd.concat(all_data, ignore_index=True)

# ▶ 5. 결과 저장
output_path = "2014_2023_계열통합_진로지표_전체통합.xlsx"
merged_df.to_excel(output_path, index=False)
print(f"✅ 전체 통합 파일 저장 완료: {output_path}")

✅ 로딩 완료: ./2014년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2015년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2016년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2017년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2018년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2019년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2020년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2021년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2022년_계열통합_진로지표추가.xlsx
✅ 로딩 완료: ./2023년_계열통합_진로지표추가.xlsx
✅ 전체 통합 파일 저장 완료: 2014_2023_계열통합_진로지표_전체통합.xlsx
